# Notebook 03 — CLIP Feature Labeling
Loads trained SAE checkpoints and labels each feature using CLIP.
Processes a subset of features (first 500 per layer) for speed.

Output: `../checkpoints/labels_layer4.pt`, `../checkpoints/labels_layer12.pt`
Each is a dict: {'labels': [...], 'scores': [...], 'top_k_indices': [...]}

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
from torchvision import datasets, transforms
from PIL import Image
from src.sae import SAE
from src.clip_labeler import label_features

DEVICE             = 'cuda'
IMAGENET_VAL_DIR   = '/path/to/imagenet/val'  # ← set this
N_IMAGES           = 10000
N_FEATURES         = 500     # label first 500 features per layer
K_PATCHES          = 10
N_SAMPLE_TOKENS    = 50000   # subsample tokens to keep RAM under 2GB

In [ ]:
pil_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
dataset = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=None)

token_to_image = {}
for img_idx in range(min(N_IMAGES, len(dataset))):
    img_path = dataset.imgs[img_idx][0]
    img = Image.open(img_path).convert('RGB')
    img = pil_transform(img)
    # map all 197 tokens of this image to the same PIL image
    for tok_offset in range(197):
        token_to_image[img_idx * 197 + tok_offset] = img

print(f'Built token_to_image with {len(token_to_image)} entries')

In [ ]:
def label_layer(acts_path, ckpt_path, save_path, layer_name):
    acts = torch.load(acts_path, weights_only=False)  # (N_tokens, 768) on CPU

    # Subsample tokens to keep memory manageable (fixed seed for reproducibility)
    idx = torch.randperm(acts.shape[0], generator=torch.Generator().manual_seed(42))[:N_SAMPLE_TOKENS]
    acts_sample = acts[idx]  # (50k, 768)

    sae = SAE(d_input=768, d_hidden=3072).cpu()
    sae.load_state_dict(torch.load(ckpt_path, weights_only=False)['state_dict'])
    sae.eval()

    with torch.no_grad():
        sae_acts, _ = sae(acts_sample)  # (50k, 3072)

    # Build token_to_image for the sampled indices
    sampled_token_to_image = {
        new_i: token_to_image[idx[new_i].item()]
        for new_i in range(len(idx))
        if idx[new_i].item() in token_to_image
    }

    results = label_features(
        sae_activations=sae_acts[:, :N_FEATURES].to(DEVICE),
        token_to_image=sampled_token_to_image,
        k=K_PATCHES,
        device=DEVICE,
    )
    torch.save(results, save_path)
    print(f'{layer_name}: labeled {len(results["labels"])} features')
    print(f'  sample labels: {results["labels"][:8]}')
    return results

In [ ]:
labels4 = label_layer(
    '../data/layer4_activations.pt',
    '../checkpoints/sae_layer4.pt',
    '../checkpoints/labels_layer4.pt',
    'layer4',
)

In [ ]:
labels12 = label_layer(
    '../data/layer12_activations.pt',
    '../checkpoints/sae_layer12.pt',
    '../checkpoints/labels_layer12.pt',
    'layer12',
)